In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.tools import InjectedToolArg
from langchain_core.messages import HumanMessage
from typing import Annotated
from dotenv import load_dotenv
import requests
import json
import os

c:\Users\sneha\Desktop\LANGCHAIN_TUTORIAL\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [4]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [5]:

multiply.name

'multiply'

In [6]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [8]:
# Defining Model

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation"
)

model_1 = ChatHuggingFace(llm=llm)

In [9]:

# tool binding

llm_with_tools = model_1.bind_tools([multiply])

In [10]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm an AI assistant here to help answer your questions. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 183, 'total_tokens': 202}, 'model_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'system_fingerprint': 'fp1-rss-s3-qwen', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d7766-3543-7970-bd01-2144f4fa70e1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 183, 'output_tokens': 19, 'total_tokens': 202})

In [11]:
llm_with_tools.invoke("Can you multiply 2 with 10")

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":2,"b":10}', 'name': 'multiply', 'description': None}, 'id': 'qwen-0-1775824685157', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 188, 'total_tokens': 213}, 'model_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'system_fingerprint': 'fp1-rss-s3-qwen', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7766-4432-7672-b629-70b9828cf38c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 10}, 'id': 'qwen-0-1775824685157', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 25, 'total_tokens': 213})

In [12]:
llm_with_tools.invoke("Can you multiply 2 with 10").tool_calls[0]

{'name': 'multiply',
 'args': {'a': 2, 'b': 10},
 'id': 'qwen-0-1775824687326',
 'type': 'tool_call'}

In [13]:
query = HumanMessage('can you multiply 7 with 1000')

In [14]:

messages = [query]

In [15]:
messages

[HumanMessage(content='can you multiply 7 with 1000', additional_kwargs={}, response_metadata={})]

In [16]:
result = llm_with_tools.invoke(messages)

In [17]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 7, 'b': 1000},
  'id': 'qwen-0-1775824689584',
  'type': 'tool_call'}]

In [18]:
messages.append(result)

In [19]:
messages

[HumanMessage(content='can you multiply 7 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":7,"b":1000}', 'name': 'multiply', 'description': None}, 'id': 'qwen-0-1775824689584', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 190, 'total_tokens': 217}, 'model_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'system_fingerprint': 'fp1-rss-s3-qwen', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7766-54c3-73f1-86cc-2903f619c54e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 1000}, 'id': 'qwen-0-1775824689584', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 27, 'total_tokens': 217})]

In [20]:
tool_result = multiply.invoke(result.tool_calls[0])

In [21]:
tool_result

ToolMessage(content='7000', name='multiply', tool_call_id='qwen-0-1775824689584')

In [22]:
messages.append(tool_result)

In [23]:
messages

[HumanMessage(content='can you multiply 7 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":7,"b":1000}', 'name': 'multiply', 'description': None}, 'id': 'qwen-0-1775824689584', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 190, 'total_tokens': 217}, 'model_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'system_fingerprint': 'fp1-rss-s3-qwen', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7766-54c3-73f1-86cc-2903f619c54e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 7, 'b': 1000}, 'id': 'qwen-0-1775824689584', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 27, 'total_tokens': 217}),
 ToolMessage(content='7000', name='multiply', tool_call_id='qwen-0-1775824689584')]

In [24]:
llm_with_tools.invoke(messages).content

'The result of multiplying 7 by 1000 is 7000.'

Currency Rate

In [25]:
# GETTING API KEY
API_KEY = os.getenv("EXCHANGERATE_API_KEY")

# tool create
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/pair/{base_currency}/{target_currency}"

  response = requests.get(url)

  return response.json()


@tool
def currency_convertor(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> dict:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    result = base_currency_value * conversion_rate
    return {
        "original_value": base_currency_value,
        "conversion_rate": conversion_rate,
        "converted_value": result
    }

In [26]:
import os 

In [27]:
currency_convertor.invoke({'base_currency_value': 10, 'conversion_rate': 91.1003})

{'original_value': 10, 'conversion_rate': 91.1003, 'converted_value': 911.003}

In [28]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation"
)

model_2 = ChatHuggingFace(llm=llm)

In [29]:

llm_with_tools = model_2.bind_tools([get_conversion_factor, currency_convertor])

In [30]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and also convert 10 USD to INR')]

In [31]:

ai_message = llm_with_tools.invoke(messages)

In [32]:
print("Tool Calls Selected:")
for tool_call in ai_message.tool_calls:
    print(f"- Tool: {tool_call['name']}, Args: {tool_call['args']}")

Tool Calls Selected:
- Tool: get_conversion_factor, Args: {'base_currency': 'USD', 'target_currency': 'INR'}
- Tool: currency_convertor, Args: {'base_currency_value': 10}


In [33]:
messages.append(ai_message)

In [34]:

messages

[HumanMessage(content='What is the conversion factor between USD and INR, and also convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_0138h6r3mclraf5k5pr7j9vd', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'currency_convertor', 'description': None}, 'id': 'call_35c7ox72j1lhu8yno2mvsfhl', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 342, 'total_tokens': 396}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7767-71ac-70f1-9995-88983259f4c5-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_0138h6r3mclraf5k5pr7j9vd', 'type

In [35]:

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'currency_convertor':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = currency_convertor.invoke(tool_call)
    messages.append(tool_message2)

In [36]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and also convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_0138h6r3mclraf5k5pr7j9vd', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'currency_convertor', 'description': None}, 'id': 'call_35c7ox72j1lhu8yno2mvsfhl', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 342, 'total_tokens': 396}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7767-71ac-70f1-9995-88983259f4c5-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_0138h6r3mclraf5k5pr7j9vd', 'type

In [37]:
response = llm_with_tools.invoke(messages)

if response.tool_calls:
    print("🔧 Tool Calls Selected:\n")
    for tool in response.tool_calls:
        print(f"Tool Name       : {tool['name']}")
        print(f"Arguments       : {tool['args']}")
        print("-" * 40)

# Final Answer Section
print("\n📊 Final Answer:\n")

content = response.content

# Clean formatting
print("=" * 50)
print(content)
print("=" * 50)


📊 Final Answer:

The current conversion factor between USD and INR is approximately 92.812. This means that 1 USD is equivalent to 92.812 INR.

To convert 10 USD to INR, the converted value is 928.12 INR.
